# 04 Incomplete engineering data

**PHASE 3 · NETWORK CONDITIONS**

What should happen when required engineering data is missing?

- Keep `source`, `derived`, `unresolved`, `documented_default`, and `ai_selected_assumption` distinct.
- Explicit user approval is required for documented defaults and AI-selected assumptions; unresolved required inputs block the user-case solve.
- The bundled IEEE13 solver demo is separate from user intake and does not depend on the optional agent bridge.


In [1]:
#@title Setup — run once, then read the results below
from hashlib import sha256
from urllib.request import urlopen

_bootstrap_url = "https://raw.githubusercontent.com/sarutesri/cept-studio-edu/main/public/notebooks/_lesson.py"
_bootstrap = urlopen(_bootstrap_url, timeout=60).read()
if sha256(_bootstrap).hexdigest() != "aa5805c306987984ba7ee64d57763db1938cb06052cf80e0f8f26fa84efba30d":
    raise ValueError("Lesson helper hash mismatch")
exec(compile(_bootstrap, "cept-lesson", "exec"), globals())


CEPT_WHEEL_URL not supplied; using the existing installed environment.
CLI: cept --version


cept-power-studio 0.2.0.dev0
Lesson helpers ready. Stage cells below run the same CEPT commands as a normal terminal.


## ⚙ Setup — Check CEPT like a terminal user

Run the same CEPT command here as in a terminal; remove the leading `!` outside Colab.

In [2]:
!cept capability show --format text

CEPT studies you can try
----------------------------
Teaching examples (not project approval)
  Fault study (OpenDSS)
  Hosting capacity (OpenDSS)
  Load flow (OpenDSS)
  Unbalanced load flow (OpenDSS)

What this means
  These are solver-backed teaching examples.
  They do NOT approve a real project or field installation.

Next
  cept environment check --format text


### ✓ Verify — Check the teaching environment

Run the readiness check before reviewing Case information.

In [3]:
!cept environment check --format text

CEPT environment check: READY
-----------------------------
Solver             OpenDSS (ready)
Version            OpenDSS 0.14.5

What this means
  CEPT and OpenDSS are installed and can run a teaching study.
  This checks the teaching environment, not a real electrical system.

Next
  cept capability show --format text


## 🧩 Inputs — Record the Case information you have

Edit only the box below; intake may be incomplete, and source-backed research inputs must stay separate from teaching assumptions.

| Policy | Allowed resolution |
| --- | --- |
| `strict` | source/derived values only |
| `assisted` | an explicitly approved, named `documented_default` |
| `exploratory` | an explicitly approved `ai_selected_assumption`; demonstrator only |

Every policy blocks unresolved required inputs. Record approval and resolution status in the ledger.


In [4]:
POLICY = "assisted"  # strict | assisted | exploratory
CASE_INFO = """
11 kV feeder
Transformer rating: 5 MVA
Load: approximately 3 MW
PV: 1 MW
Transformer impedance: not available yet
""".strip()
Path("case-info.txt").write_text(CASE_INFO + "\n", encoding="utf-8")
print("Your case information is saved")
print("------------------------------")
print(f"Review policy: {POLICY}")
print("Saved to: case-info.txt")
print("Next: review what is known and missing below")


Your case information is saved
------------------------------
Review policy: assisted
Saved to: case-info.txt
Next: review what is known and missing below


## 🔎 Review — Inspect known and missing fields

Review the ledger before building a solver Case; leave missing transformer impedance `unresolved` and do not insert a typical value.


In [5]:
resolution_payload = {
    "schema": "cept-case-info-resolution-v1",
    "policy": POLICY,
    "items": [
        {"field": "network.nominal_kv", "status": "source", "value": 11.0, "source_ref": "case-info.txt"},
        {"field": "transformer.rating_mva", "status": "source", "value": 5.0, "source_ref": "case-info.txt"},
        {"field": "load.p_mw", "status": "source", "value": 3.0, "source_ref": "case-info.txt"},
        {"field": "pv.p_mw", "status": "source", "value": 1.0, "source_ref": "case-info.txt"},
        {"field": "transformer.impedance_pu", "status": "unresolved"},
    ],
    "notes": ["Teaching intake only; no user Case has been solved."],
}
_ = Path("case-info-resolution.json").write_text(
    json.dumps(resolution_payload, indent=2) + "\n",
    encoding="utf-8",
)
print("Review list saved to case-info-resolution.json")


Review list saved to case-info-resolution.json


### ✓ Verify — Confirm the intake verdict

Validate the typed ledger (or raw-status fallback) and read the blocker list; BLOCKED is expected while required inputs remain unresolved.


In [6]:
schema_status = "available"
try:
    from cept.schema import CaseInfoResolutionLedger
    ledger = CaseInfoResolutionLedger.model_validate(resolution_payload)
    blockers = ledger.blockers(research=False)
except ImportError:
    schema_status = "upgrade wheel for typed validation"
    blockers = [item for item in resolution_payload["items"] if item["status"] == "unresolved"]
known = len(resolution_payload["items"]) - len(blockers)
missing_names = []
for blocker in blockers:
    field = getattr(blocker, "field", None)
    if field is None and isinstance(blocker, dict):
        field = blocker.get("field")
    missing_names.append(str(field or blocker))
print("Case information review - BLOCKED (this is expected)")
print("----------------------------------------------------")
print(f"Policy        {POLICY}")
print(f"Known         {known} of {len(resolution_payload['items'])} (from case-info.txt)")
print(f"Missing       {len(blockers)}: {', '.join(missing_names)}")
print("Why it matters: the transformer impedance sets the voltage drop,")
print("  so CEPT will not solve your case until it is provided or approved.")
print(f"Ledger check  {schema_status}")
print()
print("Next")
print("  add the missing value from a source, or explicitly approve a fallback,")
print("  then review again before solving")
assert blockers, "the teaching intake intentionally keeps transformer impedance unresolved"


Case information review - BLOCKED (this is expected)
----------------------------------------------------
Policy        assisted
Known         4 of 5 (from case-info.txt)
Missing       1: transformer.impedance_pu
Why it matters: the transformer impedance sets the voltage drop,
  so CEPT will not solve your case until it is provided or approved.
Ledger check  available

Next
  add the missing value from a source, or explicitly approve a fallback,
  then review again before solving


### ⚠ Resolve or approve — Choose the next action

For `transformer.impedance_pu`, seek a nameplate/test report, manufacturer datasheet, utility model, or original network model before using a demonstrator fallback.

- `assisted`: explicitly approve a named, versioned `documented_default`.
- `exploratory`: explicitly approve an `ai_selected_assumption` and record the reason; neither choice is measured or project evidence.


## ◇ Optional bridge — Organize intake only

Keep this cell collapsed unless intentionally using a configured provider key. The bridge can explain blockers but never creates evidence; the bundled solver demo runs without OpenCode, a key, or an agent command.

In [7]:
#@title Advanced — OpenCode provider bridge (optional)
AGENT_COMMAND = r'''---
description: Review Case information without inventing engineering evidence
---
Work only in the current teaching workspace. Review the learner objective plus case-info.txt and case-info-resolution.json. Accept mixed local source formats as intake evidence, but never pretend every format has a deterministic CEPT parser. Use source, derived, unresolved, documented_default, and ai_selected_assumption exactly. Never infer approval from silence. strict allows source/derived only; assisted may use an explicitly approved named default; exploratory may also use an explicitly approved AI-selected demonstrator assumption. If anything required remains unresolved, explain what is missing, why it matters, where to obtain it, and stop before the solver. CEPT Public uses noun+verb CLI; never invent commands. WORKFLOW_VALIDATED is not PROJECT_VALIDATED.

$ARGUMENTS
'''

def _colab_secret():
    names = ('OPENAI_API_KEY', 'ANTHROPIC_API_KEY')
    for name in names:
        if os.environ.get(name, '').strip():
            return name
    try:
        from google.colab import userdata
    except ImportError:
        return ''
    for name in names:
        try:
            value = (userdata.get(name) or '').strip()
        except Exception:
            continue
        if value:
            os.environ[name] = value
            return name
    return ''

def review_with_opencode():
    in_colab = 'google.colab' in sys.modules or bool(os.environ.get('COLAB_RELEASE_TAG'))
    key_name = _colab_secret() if in_colab else ''
    if not in_colab or not key_name:
        print('Optional agent: skipped outside interactive Colab or without a configured provider key.')
        return None
    opencode = Path.home() / '.opencode' / 'bin' / 'opencode'
    if not opencode.is_file():
        subprocess.run('curl -fsSL https://opencode.ai/install | bash', shell=True, check=True)
    command_dir = Path.cwd() / '.opencode' / 'commands'
    command_dir.mkdir(parents=True, exist_ok=True)
    (command_dir / 'cept.md').write_text(AGENT_COMMAND, encoding='utf-8')
    prompt = f'Review my Case information using policy {POLICY}. Do not run a user-case solver while required inputs are unresolved.'
    completed = subprocess.run([str(opencode), 'run', '--command', 'cept', prompt], text=True, capture_output=True, check=False)
    print(completed.stdout)
    if completed.returncode != 0:
        raise RuntimeError(completed.stderr.strip() or 'OpenCode review failed')
    return completed.stdout

AGENT_REVIEW = review_with_opencode()

Optional agent: skipped outside interactive Colab or without a configured provider key.


## ▶ Run — Solve the bundled demonstration

The learner Case remains blocked; run and verify the separate bundled IEEE13 demo, whose PASS applies only to that run.

In [8]:
!cept study demo load-flow \
    --network ieee13 \
    --out runs/06-bounded-load-flow \
    --force \
    --format text

!cept study verify runs/06-bounded-load-flow --format text

CEPT study result: FINISHED
----------------------------
Result             Finished the balanced load flow and saved the evidence
Saved run          runs\06-bounded-load-flow
Case fingerprint   748c8026c9d6 (matches the case you ran)

What this means
  The study completed and saved solver-backed evidence.
  It does NOT approve a real project or field installation.

Next
  cept study verify runs\06-bounded-load-flow --format text


CEPT study check: PASSED
----------------------------
Study              Load flow (OpenDSS)
Case fingerprint   748c8026c9d6 (matches the case you ran)

Checked   3 groups, 14 checks, all passed
  [PASS] Case identity (4 checks)
  [PASS] Solver result (3 checks)
  [PASS] Saved evidence (7 checks)

What this means
  The saved result matches its Case, solver run, and saved evidence.
  It does NOT approve a real project or field installation.

Saved evidence     public-verification.json

For the full check list
  cept study verify . --format json


In [9]:
#@title Checks - bounded demo receipt (advanced)
verification = read(WORKSPACE / "runs" / "06-bounded-load-flow" / "public-verification.json")
assert verification["passed"] is True
assert verification["claim"] == "WORKFLOW_VALIDATED"
assert verification["study_type"] == "load_flow"
assert verification["engine"] == "opendss"
print("Lesson check: PASSED")
print("The bundled IEEE 13-node demo solved and its evidence matches.")
print("This says nothing about your incomplete case above, which stays blocked.")


Lesson check: PASSED
The bundled IEEE 13-node demo solved and its evidence matches.
This says nothing about your incomplete case above, which stays blocked.


In [ ]:
#@title Plot — bounded demo voltage by bus and phase
import matplotlib.pyplot as plt

bounded_results = read(WORKSPACE / "runs" / "06-bounded-load-flow" / "results.json")
bounded_rows = bounded_results["load_flow"]["bus_voltages"]
plot_buses = ["sourcebus", "671", "675", "680"]
phase_values = {
    phase: [next(float(r["v_pu"]) for r in bounded_rows if r["bus"].lower() == b and r["phase"] == phase)
            for b in plot_buses]
    for phase in (1, 2, 3)
}
flat = [(b, p, phase_values[p][i]) for i, b in enumerate(plot_buses) for p in (1, 2, 3)]
max_bus, max_ph, max_v = max(flat, key=lambda t: t[2])
spreads = {b: max(phase_values[p][i] for p in (1, 2, 3)) - min(phase_values[p][i] for p in (1, 2, 3))
           for i, b in enumerate(plot_buses)}

x = list(range(len(plot_buses))); width = 0.24
fig, ax = plt.subplots(figsize=(8.5, 4.5))
for off, phase, color in [(-width, 1, "#1f5b4d"), (0, 2, "#d5654e"), (width, 3, "#e9b95f")]:
    ax.bar([v + off for v in x], phase_values[phase], width, label=f"Phase {phase}", color=color, edgecolor="white")
ax.axhline(1.0, color="#17231f", linestyle="--", linewidth=1.2, label="Nominal 1.0 pu")
ax.set_xticks(x)
ax.set_xticklabels([f"{b.title()}\n\u0394={spreads[b]:.3f}" for b in plot_buses], fontsize=9)
ax.set_title("Bounded demo: voltage by bus and phase (read per phase)")
ax.set_ylabel("Voltage magnitude (pu)")
ax.set_ylim(0.90, 1.07)
ax.scatter([plot_buses.index(max_bus)], [max_v], s=90, marker="*", color="#0f172a", zorder=5)
ax.text(plot_buses.index(max_bus), max_v + 0.004, f"max {max_bus}:{max_ph} = {max_v:.4f} pu",
        ha="center", fontsize=8, color="#0f172a")
ax.legend(frameon=False, ncol=2, fontsize=8); ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
from IPython.display import display
display(fig)
plt.close(fig)


## ◇ Interpret — Separate intake from solver evidence

- Missing required data stays unresolved until source evidence or an explicitly approved fallback is recorded.
- `documented_default` and `ai_selected_assumption` remain distinct; neither becomes measured or project evidence.
- The PASS belongs to the bundled IEEE13 demo, not the incomplete Case information.

**Try it:** Replace `CASE_INFO` with your own short description, keep one required field missing, and confirm the review stays blocked. Then add the sourced value and mark it `source`.
